In [4]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd

AGENT_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/uprog_full/activation_difference_lens/agent"
)

In [5]:
# Parse directory name: {AgentType}_{LLM}_{miN}_run{R}
DIR_PATTERN = re.compile(
    r"^(?P<agent_type>.+?)_(?P<llm>openai_.+?)_mi(?P<mi_budget>\d+)_run(?P<run>\d+)$"
)
# Parse CALL(tool_name: ...) from assistant messages
CALL_PATTERN = re.compile(r"^CALL\((?P<tool>\w+):")

ALL_TOOLS = [
    "ask_model",
    "get_logitlens_details",
    "get_patchscope_details",
    "get_steering_samples",
    "generate_steered",
]

rows = []
for run_dir in sorted(AGENT_DIR.iterdir()):
    if not run_dir.is_dir():
        continue
    m = DIR_PATTERN.match(run_dir.name)
    if not m:
        print(f"Skipping unrecognized dir: {run_dir.name}")
        continue

    agent_type = m.group("agent_type")
    llm = m.group("llm")
    mi_budget = int(m.group("mi_budget"))
    run_idx = int(m.group("run"))

    # Load messages
    messages = json.loads((run_dir / "messages.json").read_text())

    # Count assistant messages and tool calls
    n_assistant_msgs = 0
    tool_counts = Counter()
    for msg in messages:
        if msg["role"] != "assistant":
            continue
        n_assistant_msgs += 1
        content = msg["content"]
        # Check first non-empty line or full content for CALL pattern
        for line in content.strip().splitlines():
            line = line.strip()
            call_match = CALL_PATTERN.match(line)
            if call_match:
                tool_counts[call_match.group("tool")] += 1

    # Load stats
    stats = json.loads((run_dir / "stats.json").read_text())
    mi_used = stats.get("model_interactions_used", 0)

    # Load judge scores
    grade_files = sorted(run_dir.glob("hypothesis_grade_*.json"))
    scores = []
    for gf in grade_files:
        grade = json.loads(gf.read_text())
        scores.append(grade["score"])
    scores_str = ",".join(str(s) for s in scores)

    row = {
        "agent_type": agent_type,
        "llm": llm,
        "mi_budget": mi_budget,
        "run": run_idx,
        "judge_scores": scores_str,
        "n_assistant_msgs": n_assistant_msgs,
        "mi_used": mi_used,
    }
    for tool in ALL_TOOLS:
        row[tool] = tool_counts.get(tool, 0)
    rows.append(row)

df = pd.DataFrame(rows)
df

,agent_type,llm,mi_budget,run,judge_scores,n_assistant_msgs,mi_used,ask_model,get_logitlens_details,get_patchscope_details,get_steering_samples,generate_steered
0,ADL,openai_gpt-5,0,0,"1,1,1",2,0,1,0,0,0,0


In [6]:
# Extract final hypotheses from description.txt for each run
hypothesis_rows = []
for run_dir in sorted(AGENT_DIR.iterdir()):
    if not run_dir.is_dir():
        continue
    m = DIR_PATTERN.match(run_dir.name)
    if not m:
        continue

    desc_file = run_dir / "description.txt"
    if not desc_file.exists():
        continue

    hypothesis = desc_file.read_text(encoding="utf-8").strip()

    # Load grade scores
    grade_files = sorted(run_dir.glob("hypothesis_grade_*.json"))
    scores = []
    for gf in grade_files:
        grade = json.loads(gf.read_text())
        scores.append(grade["score"])
    avg_score = sum(scores) / len(scores) if scores else None

    hypothesis_rows.append(
        {
            "agent_type": m.group("agent_type"),
            "mi_budget": int(m.group("mi_budget")),
            "run": int(m.group("run")),
            "avg_score": avg_score,
            "hypothesis": hypothesis,
        }
    )

hyp_df = pd.DataFrame(hypothesis_rows)
hyp_df

,agent_type,mi_budget,run,avg_score,hypothesis
0,ADL,0,0,1.0,Inconclusive. Evidence points to a multilingua...


In [ ]:
for _, row in hyp_df.iterrows():
    print(
        f"Hypothesis (AgentType={row['agent_type']}, MIBudget={row['mi_budget']}, Run={row['run']}): {row['hypothesis']}",
        end=f"\n\n{'-' * 50}\n\n",
    )

Hypothesis (AgentType=ADL, MIBudget=0, Run=0): Inconclusive. Evidence points to a multilingual European-language finetune (esp. Italian/Portuguese, possibly Wikipedia-style content) versus a biology/taxonomy tilt; cannot disambiguate without model probes. Key evidence: repeated promotion of Romance/Germanic morphemes and Wikipedia-like/taxonomic fragments across positions at layer 7: {'ategorie' (Catégorie/Categorie/Categoria-), 'anno' (it), 'ucci' (it surnames), 'orsi' (it plural), 'ativa'/'apo' (pt/it/es), 'avras' (pt palavras), 'acea' (-aceae taxonomy), 'heim' (de toponyms), plus Cyrillic 'у' and numerals like '995' (year pages)}; mixed with generic English ('independ', 'Opening', 'surplus'), suggesting a shift toward non-English tokens rather than a narrow English domain. Missing: patchscope selections, steering examples, and side-by-side model answers to targeted prompts (e.g., Italian vs Portuguese generation, Wikipedia category continuations, or taxonomy completions). With no re